# Senate Stage 1: an incumbent-specific lagged performance predictor (ablation)

Does an incumbent's own past over- or under-performance against fundamentals carry into their re-election?

$$r_{c,t} = M^{actual}_{c,t} - \widehat M^{S1}_{c,t}, \qquad
L^{inc}_{c,t} = 0.75\,r_{c,\text{recent}} + 0.25\,r_{c,\text{previous}}$$

**The lag:**
- **The candidate:** the incumbent's own earlier Senate races, matched by FEC candidate ID (`senate_fec.csv`), not the
  state's last election. Both are D-positive, so an R incumbent who over-performed carries a negative lag.
- **Two earlier races with residuals:** the 75/25 average. **One:** that residual, flagged.
- **No lag** (open seats, appointees coded open, challengers, incumbents with no earlier race in the data or none
  with a residual): a neutral 0 in the fit, flagged. The stored lag stays missing.
- **No leakage:** residuals are S1's rolling out-of-sample errors, each from a fit on earlier years only, starting
  in 2010.

**Three specifications on identical panels.** All are trained by equal-year WLS on every observed race from 2006
(S1's rows), with incumbency fixed at +2.5, and each lag version adds one persistence coefficient:
1. production **S1**;
2. **S1 + statewide lag** (the neutral version of `02_reg_lagged_performance.ipynb`);
3. **S1 + incumbent lag**.

Each is also run through the Stage 2 blend, with unchanged inputs and weights, on 2018–2024. Production is unchanged.
Code: `src/senate/regression/lagged.py` (`incumbent_lag`).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
import senate.regression as sr
from senate import stage1, stage2
from senate.regression import lagged

pd.set_option("display.width", 200)
EVAL_YEARS = [2018, 2020, 2022, 2024]
S1 = stage1.SENATE_S1
data = sr.load_data()
nominees = pd.read_csv(ROOT / "data/processed/senate_fec.csv")

residuals = lagged.oos_residuals(stage1, S1, data, range(lagged.FIRST_RESIDUAL_YEAR, 2026, 2))
state_lag = lagged.lagged_performance(data, lagged.state_year_residuals(data[data["year"] <= 2024], residuals))
inc_lag = lagged.incumbent_lag(data, nominees, residuals, weights=lagged.WEIGHTS)
data_l = (data.merge(state_lag[[*sr.KEY, lagged.LAG, "lag_available"]], on=sr.KEY, validate="one_to_one")
          .merge(inc_lag[[*sr.KEY, lagged.INCUMBENT_LAG, "incumbent_lag_available", "inc_lag_flag"]], on=sr.KEY,
                 validate="one_to_one"))

## 1. Coverage of the incumbent lag

In [2]:
flags = inc_lag.assign(flag=inc_lag["inc_lag_flag"].str.replace(r" \(\d{4}\)", "", regex=True))
pd.crosstab(flags["year"], flags["flag"])

flag,neutral: earlier races without a residual,neutral: no earlier race of the incumbent in the data,neutral: open seat (no incumbent),one election,two elections
year,,,,,
2006,0,28,5,0,0
2008,0,28,7,0,0
2010,0,21,16,0,0
2012,18,1,12,2,0
2014,23,2,10,1,0
2016,2,1,5,26,0
2018,0,0,5,28,2
2020,0,1,6,27,1
2022,1,0,8,10,16


In [3]:
display(Markdown("**Incumbents left neutral, 2012–2024** (no earlier race in the data, or none with a residual)"))
neutral_inc = flags[flags["year"].between(2012, 2024) & flags["flag"].str.contains("earlier", regex=False)]
display(neutral_inc[["year", "state", "class", "incumbent_name", "inc_prev_years", "flag"]])
display(Markdown("**Examples, 2024**"))
inc_lag[inc_lag["year"] == 2024].dropna(subset=["incumbent_id"]).head(8).round(2)

**Incumbents left neutral, 2012–2024** (no earlier race in the data, or none with a residual)

,year,state,class,incumbent_name,inc_prev_years,flag
106,2012,CA,1,"FEINSTEIN, DIANNE",2006,neutral: earlier races without a residual
108,2012,DE,1,"CARPER, THOMAS RICHARD (TOM)",2006,neutral: earlier races without a residual
109,2012,FL,1,"NELSON, CLARENCE W. (BILL)",2006,neutral: earlier races without a residual
112,2012,MA,1,"BROWN, SCOTT P.",NaN,neutral: no earlier race of the incumbent in t...
113,2012,MD,1,"CARDIN, BENJAMIN L. (BEN)",2006,neutral: earlier races without a residual
115,2012,MI,1,"STABENOW, DEBORAH ANN (DEBBIE)",2006,neutral: earlier races without a residual
116,2012,MN,1,"KLOBUCHAR, AMY J.",2006,neutral: earlier races without a residual
117,2012,MO,1,"MCCASKILL, CLAIRE",2006,neutral: earlier races without a residual
118,2012,MS,1,"WICKER, ROGER F.",2008,neutral: earlier races without a residual
119,2012,MT,1,"TESTER, JON",2006,neutral: earlier races without a residual


**Examples, 2024**

,year,state,class,incumbent_id,incumbent_name,inc_prev_years,inc_r_recent,inc_r_previous,incumbent_lag,inc_lag_n,inc_lag_flag,incumbent_lag_available
315,2024,CT,1,S2CT00132,"MURPHY, CHRISTOPHER SCOTT (CHRIS)","2012,2018",-3.25,-4.45,-3.55,2,two elections,True
317,2024,FL,1,S8FL00273,"SCOTT, RICHARD (RICK)",2018,-8.10,NaN,-8.10,1,one election (2018),True
318,2024,HI,1,S2HI00106,"HIRONO, MAZIE K.","2012,2018",-3.09,-6.43,-3.92,2,two elections,True
320,2024,MA,1,S2MA00170,"WARREN, ELIZABETH A.","2012,2018",-11.28,-12.23,-11.52,2,two elections,True
322,2024,ME,1,S2ME00109,"KING, ANGUS S., JR.","2012,2018",6.36,13.65,8.18,2,two elections,True
324,2024,MN,1,S6MN00267,"KLOBUCHAR, AMY J.","2012,2018",13.09,26.96,16.56,2,two elections,True
325,2024,MO,1,S8MO00160,"HAWLEY, JOSH",2018,2.30,NaN,2.30,1,one election (2018),True
326,2024,MS,1,S8MS00196,"WICKER, ROGER F.","2012,2018",-6.69,-0.44,-5.13,2,two elections,True


## 2. Fits and persistence coefficients

In [4]:
s1_pred = pd.concat([r.predictions for r in stage1.rolling_predictions(S1, eval_years=EVAL_YEARS, data=data).values()],
                    ignore_index=True)
common = dict(data=data_l, s1_formula=S1.formula, fixed=dict(S1.fixed), s1_predictions=s1_pred,
              eval_years=EVAL_YEARS, missing_lag="neutral")
state_pred, state_coef = lagged.fit_predict_lagged(**common, lag=lagged.LAG, available="lag_available")
inc_pred, inc_coef = lagged.fit_predict_lagged(**common, lag=lagged.INCUMBENT_LAG,
                                               available="incumbent_lag_available")
PREDICTIONS = {"S1": s1_pred, "S1 + statewide lag": state_pred, "S1 + incumbent lag": inc_pred}
for name, coef in [("S1 + statewide lag", state_coef), ("S1 + incumbent lag", inc_coef)]:
    display(Markdown(f"**{name}** (`n_train_with_L`: training races with a real lag; the rest use 0)"))
    display(coef)
# identical panels: same races predicted, and every model trained on S1's rows
keys = {name: tuple(map(tuple, p[sr.KEY].to_numpy())) for name, p in PREDICTIONS.items()}
assert len(set(keys.values())) == 1

**S1 + statewide lag** (`n_train_with_L`: training races with a real lag; the rest use 0)

,Intercept,partisan_lean,gen_margin_raw,lagged_performance,inc_signed,n_train,n_train_with_L,train_years
eval_year,,,,,,,,
2018,1.564085,0.985278,0.854889,0.462803,2.5,198,31,2006-2016
2020,1.590079,0.951381,0.880695,0.136443,2.5,232,65,2006-2018
2022,1.42247,0.961413,0.854194,0.112614,2.5,266,96,2006-2020
2024,1.709898,0.964507,0.83193,0.13773,2.5,299,127,2006-2022


**S1 + incumbent lag** (`n_train_with_L`: training races with a real lag; the rest use 0)

,Intercept,partisan_lean,gen_margin_raw,incumbent_lag,inc_signed,n_train,n_train_with_L,train_years
eval_year,,,,,,,,
2018,2.621648,0.968974,0.826617,0.542128,2.5,198,29,2006-2016
2020,2.225244,0.939042,0.805586,0.482011,2.5,232,58,2006-2018
2022,2.068021,0.949682,0.788823,0.456677,2.5,266,85,2006-2020
2024,2.349631,0.943238,0.759248,0.38867,2.5,299,110,2006-2022


## 3. Out-of-sample comparison, 2018–2024

**Metrics:**
- `mae` and `bias` (actual − predicted; negative means too Democratic), over all races.
- `ex_ante_mae`: races with |lean + eve generic ballot| ≤ 10.
- `incumbent_mae`: races with an incumbent running. `competitive_incumbent_mae`: incumbents in ex-ante competitive
  races.
- `incumbent_with_lag_mae`: incumbents who have a real incumbent lag.
- `accuracy_pct`: winners called correctly.

In [5]:
polls, ratings = stage2.load_inputs()
blends = {name: stage2.blend(p, polls, ratings) for name, p in PREDICTIONS.items()}
COLUMNS = {"fundamentals": "margin_fundamentals", "blend": "prediction"}
rows = blends["S1"][sr.KEY].merge(
    data_l[[*sr.KEY, "partisan_lean", "gb_margin_2p_norm", "inc_signed", "incumbent_lag_available"]], on=sr.KEY,
    validate="one_to_one")
EX_ANTE = ((rows["partisan_lean"] + rows["gb_margin_2p_norm"]).abs() <= sr.EX_ANTE_THRESHOLD).to_numpy()
INCUMBENT = (rows["inc_signed"] != 0).to_numpy()
WITH_LAG = rows["incumbent_lag_available"].to_numpy()


def metrics(b, column, mask=None):
    mask = np.ones(len(b), dtype=bool) if mask is None else mask
    actual, pred = b["actual"].to_numpy()[mask], b[column].to_numpy()[mask]
    err, ea, inc, wl = actual - pred, EX_ANTE[mask], INCUMBENT[mask], WITH_LAG[mask]
    called = (pred != 0) & (actual != 0)
    return pd.Series({
        "n": len(err), "mae": np.abs(err).mean(), "bias": err.mean(), "ex_ante_n": int(ea.sum()),
        "ex_ante_mae": np.abs(err[ea]).mean(), "incumbent_n": int(inc.sum()), "incumbent_mae": np.abs(err[inc]).mean(),
        "competitive_incumbent_n": int((inc & ea).sum()), "competitive_incumbent_mae": np.abs(err[inc & ea]).mean(),
        "incumbent_with_lag_mae": np.abs(err[inc & wl]).mean(),
        "accuracy_pct": 100 * (np.sign(pred[called]) == np.sign(actual[called])).mean()})


for b in blends.values():  # every blend row is in the same order as `rows`
    assert b[sr.KEY].equals(blends["S1"][sr.KEY])
pooled = pd.DataFrame({(stage, name): metrics(b, column) for stage, column in COLUMNS.items()
                       for name, b in blends.items()}).T
pooled.index.names = ["stage", "model"]
pooled.round(2)

n   mae  bias  ex_ante_n  ex_ante_mae  incumbent_n  incumbent_mae  competitive_incumbent_n  competitive_incumbent_mae  incumbent_with_lag_mae  accuracy_pct
stage        model                                                                                                                                                                              
fundamentals S1                  134.0  5.26 -0.15       39.0         3.97        105.0           5.17                     31.0                       4.34                    5.20         93.28
             S1 + statewide lag  134.0  5.73  0.13       39.0         4.32        105.0           5.73                     31.0                       4.78                    5.76         91.79
             S1 + incumbent lag  134.0  5.41 -0.92       39.0         3.78        105.0           5.33                     31.0                       4.21                    5.35         93.28
blend        S1                  134.0  4.21 -1.33       39.0         3.13        105.0           4.27                     31.0                       3.41                    4.25         93.28
             S1 + statewide lag  134.0  4.34 -1.22       39.0         3.02        105.0           4.43                     31.0                       3.33                    4.41         93.28
             S1 + incumbent lag  134.0  4.32 -1.66       39.0         3.15        105.0           4.39                     31.0                       3.45                    4.37         95.52

### By election year

In [6]:
by_year = []
for stage, column in COLUMNS.items():
    for name, b in blends.items():
        for year in EVAL_YEARS:
            m = metrics(b, column, (b["year"] == year).to_numpy())
            by_year.append({"stage": stage, "model": name, "year": year, **m.to_dict()})
by_year = pd.DataFrame(by_year).set_index(["stage", "year", "model"])
by_year[["mae", "bias", "ex_ante_mae", "incumbent_mae", "competitive_incumbent_mae", "accuracy_pct"]].round(2)

mae  bias  ex_ante_mae  incumbent_mae  competitive_incumbent_mae  accuracy_pct
stage        year model                                                                                              
fundamentals 2018 S1                  6.39  1.74         4.66           6.60                       5.55         91.18
             2020 S1                  4.98 -2.73         3.13           5.05                       3.59         97.06
             2022 S1                  5.17  1.41         4.67           4.67                       4.60         90.91
             2024 S1                  4.49 -1.00         3.60           4.06                       3.86         93.94
             2018 S1 + statewide lag  8.53  2.61         6.28           8.97                       7.53         88.24
             2020 S1 + statewide lag  4.97 -2.62         3.03           4.98                       3.52         97.06
             2022 S1 + statewide lag  5.04  1.56         4.50           4.50                       4.38         90.91
             2024 S1 + statewide lag  4.30 -1.00         3.76           3.89                       4.06         90.91
             2018 S1 + incumbent lag  6.62 -1.99         4.34           6.89                       5.40         88.24
             2020 S1 + incumbent lag  4.76 -2.10         3.36           4.74                       3.88        100.00
             2022 S1 + incumbent lag  6.01  2.77         4.76           5.92                       4.94         90.91
             2024 S1 + incumbent lag  4.22 -2.31         2.90           3.45                       3.02         93.94
blend        2018 S1                  3.38  0.44         3.01           3.73                       3.31         91.18
             2020 S1                  5.13 -4.34         2.93           5.36                       3.35         94.12
             2022 S1                  4.28  0.88         3.94           4.08                       3.89         90.91
             2024 S1                  4.06 -2.27         2.76           3.83                       3.17         96.97
             2018 S1 + statewide lag  3.86  0.81         2.61           4.31                       3.02         91.18
             2020 S1 + statewide lag  5.25 -4.31         2.78           5.48                       3.18         94.12
             2022 S1 + statewide lag  4.21  0.94         3.90           3.99                       3.84         90.91
             2024 S1 + statewide lag  4.05 -2.28         2.86           3.76                       3.29         96.97
             2018 S1 + incumbent lag  3.91 -1.08         3.31           4.37                       3.79         97.06
             2020 S1 + incumbent lag  4.85 -4.14         2.83           4.97                       3.19         97.06
             2022 S1 + incumbent lag  4.49  1.57         4.05           4.43                       4.12         90.91
             2024 S1 + incumbent lag  4.02 -2.94         2.58           3.67                       2.89         96.97

### Blend: polled vs unpolled races

In [7]:
for polled in [True, False]:
    mask = blends["S1"]["polling_available"].to_numpy() == polled
    display(Markdown(f"**{'Polled' if polled else 'Unpolled'}** ({int(mask.sum())} races)"))
    display(pd.DataFrame({name: metrics(b, "prediction", mask) for name, b in blends.items()}).T
            [["n", "mae", "bias", "ex_ante_mae", "incumbent_mae", "competitive_incumbent_mae"]].round(2))

**Polled** (87 races)

,n,mae,bias,ex_ante_mae,incumbent_mae,competitive_incumbent_mae
S1,87.0,4.00,-1.55,3.19,4.03,3.39
S1 + statewide lag,87.0,4.08,-1.41,3.04,4.15,3.27
S1 + incumbent lag,87.0,3.96,-1.91,3.25,3.99,3.50


**Unpolled** (47 races)

,n,mae,bias,ex_ante_mae,incumbent_mae,competitive_incumbent_mae
S1,47.0,4.61,-0.93,2.52,4.69,3.65
S1 + statewide lag,47.0,4.83,-0.87,2.81,4.91,4.18
S1 + incumbent lag,47.0,4.99,-1.20,1.99,5.10,2.75


## Conclusion

**Incumbent strength persists, but adding it doesn't improve the forecast. It isn't adopted.** Production S1 is
unchanged.

**1. The signal is real.** The incumbent lag's persistence coefficient is 0.54, 0.48, 0.46 and 0.39 across the four
windows, steady at about 0.4–0.5. The statewide lag's settles at 0.11–0.14. An incumbent who beat fundamentals by
10 points last time is predicted about 4–5 points better than fundamentals this time.

**2. Out of sample it doesn't pay.** Pooled 2018–2024, same 134 races:

| | S1 | S1 + statewide lag | S1 + incumbent lag |
|---|---|---|---|
| Fundamentals MAE / bias | **5.26** / −0.15 | 5.73 / +0.13 | 5.41 / −0.92 |
| Fundamentals ex-ante / competitive-incumbent MAE | 3.97 / 4.34 | 4.32 / 4.78 | **3.78** / **4.21** |
| Blend MAE / bias | **4.21** / −1.33 | 4.34 / −1.22 | 4.32 / −1.66 |
| Blend ex-ante / incumbent / competitive-incumbent MAE | 3.13 / **4.27** / 3.41 | **3.02** / 4.43 / **3.33** | 3.15 / 4.39 / 3.45 |
| Blend winners called | 93.3% | 93.3% | **95.5%** |

- **The fundamentals split:** better on competitive races and competitive incumbents, worse overall and on all
  incumbents. The bias moves 0.8 points toward Democrats.
- **The blend is mostly a wash:** MAE about +0.1 everywhere, and the bias toward Democrats grows. It calls 3 more
  winners out of 134, which is within noise.
- **The years disagree.** In the fundamentals, the incumbent lag helps 2020 (4.76 vs 4.98) and 2024 (4.22 vs 4.49;
  competitive incumbents 3.02 vs 3.86), and hurts 2018 (6.62 vs 6.39) and 2022 (6.01 vs 5.17). No consistent edge.
- **Polled races:** the blend is about level (3.96 vs 4.00), since the polls already reflect candidate strength.
  **Unpolled races:** it's worse (4.99 vs 4.61).

**Why it doesn't pay:**
- Most of an incumbent's past over-performance is already in the polls and ratings that the blend adds.
- Past residuals also reflect conditions that change, such as the opponent and the cycle.
- With 4 test years and about 30 competitive incumbents, a real but modest effect can't be shown to beat noise.

**Limitations:**
- Residuals start in 2010. Before 2016, most incumbents' earlier race is too early to have a residual.
- Four incumbents who won off-cycle specials (MA 2010 and 2013, NJ 2013, AL 2017), and Murkowski's 2010 write-in, have
  no matched earlier race and are left neutral.
- Weights fixed at 75/25, not tuned.
- Worth re-checking once 2026 results add another cycle of incumbent lags.
